# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [ ]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [ ]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [ ]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [ ]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [ ]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [ ]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [ ]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [ ]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [ ]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [8]:
from typing import Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()

        self.forget_gate = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size, bias=False),
            nn.Sigmoid()
        )

        self.candidate_gate = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size, bias=False),
            nn.Tanh()
        )

        self.input_gate = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size, bias=False),
            nn.Sigmoid()
        )

        self.output_gate = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor, h: torch.Tensor, c: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:

        # x: [B, D1]
        # h, c: [B, D2]

        xh = torch.cat((x, h), dim=-1) # [B, D1+D2]
        forget_gate_out = self.forget_gate(xh) # [B, D2]

        c = forget_gate_out * c # [B, D2]

        candidate_gate_out = self.candidate_gate(xh) # [B, D2]
        input_gate_out = self.input_gate(xh) # [B, D2]

        c = candidate_gate_out * input_gate_out + c

        output_gate_out = self.output_gate(xh) # [B, D2]

        h = output_gate_out * F.tanh(c)

        return h, c

cell = LSTMCell(8 ,16)
x = torch.rand(5, 16, 8)

h = torch.zeros(5, 16)
c = torch.zeros(5, 16)

for x_seg in torch.chunk(x, 16, dim=1):
    x_seg = x_seg.squeeze(dim=1)
    h, c = cell(x_seg, h, c)

# torch.save(cell.state_dict(), 'my_lstm.pt')
print(cell.state_dict())

checkpoint_state = {
    'state_dict': cell.state_dict(),
}

# torch.save(checkpoint_state, 'my_checkpoint.ckpt')

OrderedDict({'forget_gate.0.weight': tensor([[-0.0044,  0.0492, -0.1348,  0.1224,  0.0313,  0.0358, -0.1388, -0.0014,
          0.1688,  0.2021,  0.1517,  0.0847,  0.1182,  0.1817, -0.1746, -0.1452,
         -0.0906, -0.0168,  0.1232, -0.0074, -0.1634,  0.0926,  0.0438, -0.0580],
        [-0.1988,  0.1099, -0.1052, -0.0482, -0.0536, -0.0953,  0.0483, -0.0788,
         -0.0140, -0.1624,  0.0191, -0.1546, -0.1592, -0.0702, -0.0568, -0.0822,
         -0.1900, -0.1712, -0.0454,  0.1712, -0.1897, -0.0173,  0.0764, -0.0008],
        [ 0.1587,  0.1011,  0.1660, -0.1458,  0.1852, -0.0260,  0.1962,  0.1688,
          0.1634, -0.1762,  0.1309,  0.0963, -0.1732, -0.1088,  0.1197, -0.0101,
          0.1204,  0.0248,  0.1555, -0.0376,  0.1087, -0.1285, -0.0282, -0.0012],
        [ 0.1131, -0.1496, -0.0716, -0.0970,  0.1151,  0.0653,  0.0293, -0.0670,
         -0.1875, -0.1571,  0.1114,  0.0997,  0.0186,  0.0646, -0.1851,  0.1465,
          0.1085,  0.1169,  0.0776,  0.0817,  0.0653,  0.1818, -0.133

### Inception

![inception](assets/inception.png)

In [9]:
class InceptionLayer(nn.Module):
    def __init__(self, in_channels: int = 8, branch1_out: int = 4,
                 branch2_hidden: int = 8, branch2_out: int = 4,
                 branch3_hidden: int = 8, branch3_out: int = 4,
                 branch4_out: int = 4):
        super().__init__()

        self.branch1 = nn.Conv2d(in_channels, branch1_out, 1)

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, branch2_hidden, 1),
            nn.Conv2d(branch2_hidden, branch2_out, 3, padding=1),
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, branch3_hidden, 1),
            nn.Conv2d(branch3_hidden, branch3_out, 5, padding=2),
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, branch4_out, 1),
        )

    def forward(self, x: torch.Tensor):
        # x: [B, C, H, W]

        branch1_out = self.branch1(x)
        branch2_out = self.branch2(x)
        branch3_out = self.branch3(x)
        branch4_out = self.branch4(x)

        return torch.cat([branch1_out, branch2_out, branch3_out, branch4_out], dim=1)

x = torch.rand(6, 8, 16, 16)

layer = InceptionLayer(8,)
layer(x).shape

torch.Size([6, 16, 16, 16])

### SE

![se](assets/SqueezeAndExcite.png)

In [10]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()

        # F_sq
        self.squeeze = nn.AdaptiveAvgPool2d(1)

        # F_ex
        self.excite = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, _, _ = x.shape

        s = self.squeeze(x).view(B, C)  # [B, C]
        s = self.excite(s)              # [B, C]
        s = s.view(B, C, 1, 1)          # [B, C, 1, 1]

        # F_scale
        return x * s


se = SEBlock(16)
se(torch.randn(2, 16, 8, 8)).shape

torch.Size([2, 16, 8, 8])

### Selective Kernel

![selective](assets/SelectiveKernel.png)


### PatchMerger

![patchmerger](assets/PatchMerger.png)


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [11]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()


    def forward(self, x):
        y = self.fc1(x)
        y = self.relu(y)
        y = self.fc2(y)
        y = y + x

        return self.relu(y)


block = ResidualBlock(16)
block(torch.randn(4, 16)).shape

torch.Size([4, 16])

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [12]:
class SeparableConv2d(nn.Module):
    def __init__(self, kernel_size, M, N):
        super().__init__()

        self.conv1 = nn.Conv2d(M, M, kernel_size, padding=kernel_size // 2, groups=M)
        self.conv2 = nn.Conv2d(M, N, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)

        return x


conv = SeparableConv2d(3, 8, 16)
conv(torch.randn(2, 8, 32, 32)).shape

torch.Size([2, 16, 32, 32])

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [13]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.Wv = nn.Linear(d_model, d_model)
        self.Wq = nn.Linear(d_model, d_model)
        self.Wa = nn.Linear(d_model, d_model)

    def forward(self, q, k):
        # q: [B, d], k: [B, L, d]
        score = torch.bmm(k, self.Wa(q).unsqueeze(-1)).squeeze(-1) # [B, L]
        att = torch.softmax(score, dim=1) # [B, L]
        context = (att.unsqueeze(-1) * k).sum(dim=1) # [B, d]
        out = torch.tanh(self.Wv(context) + self.Wq(q)) # [B, d]

        return out


attn = VanillaAttention(16)
attn(torch.randn(4, 16), torch.randn(4, 10, 16)).shape

torch.Size([4, 16])

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [14]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V):
        # Q: [B, Lq, d], K: [B, Lk, d], V: [B, Lk, d]
        d_sqrt = Q.shape[-1] ** 0.5

        scores = (Q @ K.transpose(1, 2)) / d_sqrt # [B, Lq, Lk]
        att = torch.softmax(scores, dim=-1) @ V # [B, Lq, d]

        return att


sdpa = ScaledDotProductAttention()
sdpa(torch.randn(2, 5, 16), torch.randn(2, 7, 16), torch.randn(2, 7, 16)).shape

torch.Size([2, 5, 16])

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x = [B, S, d_model]

        B = x.shape[0]
        S = x.shape[1]

        Q = self.Wq(x) # [B, S, d]
        K = self.Wk(x) # [B, S, d]
        V = self.Wv(x) # [B, S, d]

        Q = Q.reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2) # [B, num_heads, S, head_dim]
        K = K.reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2) # [B, num_heads, S, head_dim]
        V = V.reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2) # [B, num_heads, S, head_dim]

        d_sqrt = self.head_dim ** 0.5

        score = (Q @ K.transpose(-1, -2)) / d_sqrt # [B, num_heads, S, S]
        att = torch.softmax(score, dim=-1) @ V # [B, num_heads, S, head_dim]

        att = att.transpose(1, 2) # [B, S, num_heads, head_dim]
        att = att.reshape(B, S, self.d_model)

        att = self.Wo(att)

        return att


mha = MultiHeadAttention(32, 4)
mha(torch.randn(2, 10, 32)).shape

torch.Size([2, 10, 32])

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [16]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        # из предыдущего задания
        self.attention = MultiHeadAttention(d_model, num_heads)

        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.drop2 = nn.Dropout(dropout)

        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)

    def FeedForward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)

        return x

    def forward(self, x):
        y = self.layernorm1(x)
        y = self.attention(y)
        y = y + x
        z = self.layernorm2(y)
        z = self.FeedForward(z)
        z = z + y

        return z


enc = TransformerEncoderLayer(32, 4)
enc(torch.randn(2, 10, 32)).shape

torch.Size([2, 10, 32])

## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [17]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()

        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches, channels, token_hidden, channel_hidden):
        super().__init__()

        self.norm1 = nn.LayerNorm(channels)
        self.token_mlp = MLP(num_patches, token_hidden)

        self.norm2 = nn.LayerNorm(channels)
        self.channel_mlp = MLP(channels, channel_hidden)

    def forward(self, x):
        # x: [B, patches, channels]
        y = self.norm1(x)
        y = y.transpose(1, 2) # [B, channels, patches]
        y = self.token_mlp(y)
        y = y.transpose(1, 2) # [B, patches, channels]
        x = x + y

        y = self.norm2(x)
        y = self.channel_mlp(y)
        x = x + y

        return x


mixer = MLPMixerBlock(num_patches=16, channels=32, token_hidden=64, channel_hidden=128)
mixer(torch.randn(2, 16, 32)).shape

torch.Size([2, 16, 32])

## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [18]:
class ConvMixer(nn.Module):

    def __init__(self, dim, kernel_size=9):
        super().__init__()

        self.depthwise = nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim)
        self.act1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(dim)

        self.pointwise = nn.Conv2d(dim, dim, 1)
        self.act2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(dim)

    def forward(self, x):
        # x: [B, dim, H, W]
        y = self.depthwise(x)
        y = self.act1(y)
        y = self.bn1(y)
        x = x + y

        x = self.pointwise(x)
        x = self.act2(x)
        x = self.bn2(x)

        return x


cm = ConvMixer(32)
cm(torch.randn(2, 32, 16, 16)).shape

torch.Size([2, 32, 16, 16])

## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Все три архитектуры устроены одинаково: блок = смешивание токенов (патчей) + смешивание каналов (MLP / 1x1 conv), плюс residual и нормализация. Получается, что основную пользу даёт сама эта схема (чередовать обмен информацией между патчами и обработку каждого патча), а конкретный способ смешивать токены не так важен. Attention - это просто один из вариантов смешивания, и его можно заменить на обучаемую матрицу (MLPMixer) или на свёртку (ConvMixer). Плюс большую роль играет патчификация входа.
